In [ ]:
import torch
import pandas as pd
import gradio as gr
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from tqdm import tqdm
from datasets import load_dataset
from huggingface_hub import login

In [ ]:
dataset = load_dataset("Arseney/parallel_corpus_russian_rsl_glosses")
dataset

In [ ]:
model_path = "" # Model's name
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
def tokenizing(batch):
    return tokenizer(
        batch["russian"],
        text_target=batch["rsl"],
        truncation=True,
        max_length=128,
    )

In [ ]:
tokenized_dataset = dataset.map(tokenizing, batched=True, remove_columns=dataset["train"].column_names)

In [ ]:
tokenized_dataset

In [ ]:
training_args = TrainingArguments(
    output_dir="", # output dir's name
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=2e-5,
    fp16=True,
    gradient_checkpointing=True,
    optim="adafactor",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

In [ ]:
!nvidia-smi

In [ ]:
trainer.train()

In [ ]:
model.push_to_hub("Arseniy-Polyakov/...")
tokenizer.push_to_hub("Arseniy_Polyakov/...")

In [ ]:
tokenizer_hf = AutoTokenizer.from_pretrained("Arseniy-Polyakov/...")
model_hf = AutoModelForSeq2SeqLM.from_pretrained("Arseniy-Polyakov/...")

In [ ]:
target_data = [[item["rsl"].split()] for item in dataset["test"]]
texts_for_translation = [item["russian"] for item in dataset["test"]]
texts_for_translation

In [ ]:
def translate_texts(texts, model, tokenizer, device="cpu", max_length=128, num_beams=5):
    model.to(device)
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )
    return translations

In [ ]:
translated_sentences = translate_texts(texts_for_translation, model_hf, tokenizer_hf)

In [ ]:
translated_sentences

In [ ]:
def translate_texts_for_gradio(text: str):
  inputs = tokenizer_hf(text, return_tensors="pt")

  outputs = model_hf.generate(
      **inputs,
      max_new_tokens=60,
      num_beams=4,
      early_stopping=True,
      repetition_penalty=1.1,
  )
  gloss_sentence = tokenizer_hf.decode(outputs[0], skip_special_tokens=True).strip()
  return gloss_sentence

In [ ]:
demo = gr.Interface(fn=translate_texts, inputs="text", outputs="text")
demo.launch()